# Moving MNIST SimVP

Train a CNN-only SimVP forecaster on Moving MNIST sequences.


In [ ]:
from pathlib import Path
import subprocess
import sys

GITHUB_REPO_URL = "https://github.com/jordanshivers/generative-video-forecasting.git"
REPO_DIRNAME = "generative-video-forecasting"
INSTALL_REQUIREMENTS = True
RETRAIN = False


def find_repo_root(start=None):
    current = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "requirements.txt").exists() and (candidate / "src" / "video_forecasting").exists():
            return candidate
    if Path("/content").exists():
        repo_root = Path("/content") / REPO_DIRNAME
        if not repo_root.exists():
            subprocess.run(["git", "clone", GITHUB_REPO_URL, str(repo_root)], check=True)
        return repo_root
    raise RuntimeError("Could not find the generative-video-forecasting repository root.")


REPO_ROOT = find_repo_root()
if INSTALL_REQUIREMENTS and Path("/content").exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_ROOT / "requirements.txt")], check=True)

sys.path.insert(0, str(REPO_ROOT / "src"))
print(f"Repo root: {REPO_ROOT}")


In [ ]:
import torch
from torch.utils.data import DataLoader

from video_forecasting.models.simvp import SimVP, SimVPSequenceDataset
from video_forecasting.runtime import get_data_dir, get_device, get_output_dir, set_seed
from video_forecasting.presets import batch_size_for_device, get_preset
from video_forecasting.training import count_parameters, evaluate_simvp, train_simvp_epoch
from video_forecasting.visualization import (
    display_video,
    generate_simvp_rollout_movie,
    plot_training_curves,
    set_output_dir,
    visualize_simvp_predictions,
)

set_seed(42)
device = get_device(prefer_mps=True)
DATA_DIR = get_data_dir(REPO_ROOT)
OUTPUT_DIR = get_output_dir("train_moving_mnist_simvp", REPO_ROOT)
set_output_dir(OUTPUT_DIR)
print(f"Device: {device}")
print(f"Output dir: {OUTPUT_DIR}")


## Dataset


In [ ]:
from video_forecasting.datasets.moving_mnist import MovingMNISTDataset

dataset_cfg = get_preset("moving_mnist")
simvp_cfg = get_preset("simvp")
max_sequences = dataset_cfg["max_sequences"]
sequence_length = dataset_cfg["sequence_length"]
context_frames = simvp_cfg["context_frames"]
pred_frames = simvp_cfg["pred_frames"]
batch_size = batch_size_for_device(device, simvp_cfg["batch_size"])

train_base = MovingMNISTDataset(
    root=str(DATA_DIR),
    train=True,
    sequence_length=sequence_length,
    normalize=True,
    frame_separation=dataset_cfg["frame_separation"],
    download=True,
    max_sequences=max_sequences,
)
test_base = MovingMNISTDataset(
    root=str(DATA_DIR),
    train=False,
    sequence_length=sequence_length,
    normalize=True,
    frame_separation=dataset_cfg["frame_separation"],
    download=True,
    max_sequences=max_sequences,
)
train_dataset = SimVPSequenceDataset(train_base, context_frames=context_frames, pred_frames=pred_frames, stride=1)
test_dataset = SimVPSequenceDataset(test_base, context_frames=context_frames, pred_frames=pred_frames, stride=1)

pin_memory = device.type == "cuda"
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=pin_memory)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=pin_memory)
print(f"Train windows: {len(train_dataset)}; test windows: {len(test_dataset)}")


## Model


In [ ]:
model = SimVP(
    shape_in=(context_frames, 1, 64, 64),
    hid_s=simvp_cfg["hid_s"],
    hid_t=simvp_cfg["hid_t"],
    num_spatial_layers=simvp_cfg["num_spatial_layers"],
    num_temporal_layers=simvp_cfg["num_temporal_layers"],
    kernels=(3, 5, 7, 11),
    groups=8,
).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
checkpoint_path = OUTPUT_DIR / "simvp_moving_mnist_model.pt"
print(f"Parameters: {count_parameters(model):,}")


## Train


In [ ]:
num_epochs = simvp_cfg["num_epochs"]
train_losses = []
val_losses = []

if checkpoint_path.exists() and not RETRAIN:
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    print(f"Loaded checkpoint: {checkpoint_path}")
else:
    for epoch in range(num_epochs):
        train_loss = train_simvp_epoch(model, train_loader, optimizer, device)
        val_loss = evaluate_simvp(model, test_loader, device)
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        print(f"Epoch {epoch + 1}/{num_epochs}: train={train_loss:.5f}, val={val_loss:.5f}")
    torch.save(model.state_dict(), checkpoint_path)
    plot_training_curves(
        train_losses,
        val_losses,
        output_path=OUTPUT_DIR / "simvp_training_curves.png",
        title="SimVP Training Curves",
    )
    print(f"Saved checkpoint: {checkpoint_path}")


## Evaluate

Create prediction figures, generate an autoregressive rollout movie, and display it inline.


In [ ]:
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model.eval()

visualize_simvp_predictions(
    model,
    test_dataset,
    num_samples=4,
    device=device,
    title_prefix="Moving MNIST ",
)

test_sequence_idx = 0
test_sequence = test_base.sequences[test_sequence_idx]
rollout_path = generate_simvp_rollout_movie(
    model,
    sequence=test_sequence,
    context_frames=context_frames,
    num_predictions=12,
    start_frame=0,
    dataset_type="moving_mnist",
    device=device,
    fps=10,
    output_dir=str(OUTPUT_DIR / "output_mp4s"),
)
print(f"Rollout movie saved to: {rollout_path}")
display_video(rollout_path)
